# 工具化记忆（Memory with Tools / Memory-as-a-Tool）

> **把可调用的记忆工具（`save_memory`、`search_memory`、`update_memory`、`delete_memory`）交给智能体，让它自己决定何时记住什么。**

想象一位图书管理员：她决定哪些书上架、替你查找哪些书、淘汰哪些过时的书。她不按固定流程办事，而是根据你的请求自行判断。记忆即工具（Memory-as-a-Tool）让 AI 智能体对自己的记忆拥有同样的控制权。这个类比有一点不成立：智能体还负责写书（它从对话中提炼事实）。

大多数智能体记忆系统在幕后默默运行。框架拦截每条消息，生成嵌入，再把检索到的分块塞进提示词。智能体对存什么没有发言权，不能更新过时的事实，也不能选择遗忘。

对简单的聊天机器人来说，这够用了。但一旦智能体需要有所取舍，它就失效了。比如：记住用户的饮食禁忌，同时忽略闲聊；或者更正之前保存过的事实。

**记忆即工具（Memory-as-a-Tool）**模式反转了这一模型。记忆操作变成工具 schema（对可调用函数的结构化描述）。LLM 在生成过程中自行决定是否调用 `save_memory`、`search_memory`、`update_memory` 或 `delete_memory`。它复用了驱动所有现代智能体的工具使用循环。记忆工具与网页搜索、代码执行等其他工具并列。

本 notebook 演示如何用 **Anthropic SDK**（Anthropic 用于调用 Claude 的 Python 库）从零构建这套系统。你将定义四个记忆工具，搭建一个基于向量的记忆库，并接好整个智能体工具使用循环。

**读完本 notebook，你将理解：**
- 如何把记忆操作定义成工具 schema。
- 如何搭建一个带基于嵌入的语义检索的记忆库。
- 智能体循环如何分发工具调用并回传结果。
- 智能体自主记忆何时优于隐藏在流程（pipeline）里的记忆。


## 核心概念

- **工具 schema（tool schema）**：对可调用函数的结构化描述，包含工具名称、用途和期望的输入（采用 JSON Schema）。LLM 读取这些 schema，决定调用哪些工具。
- **记忆 CRUD**：管理存储数据的四个核心操作。CRUD 指创建（Create）、读取（Read）、更新（Update）、删除（Delete）。这里我们把它们暴露为 `save_memory`、`search_memory`、`update_memory` 和 `delete_memory`。
- **智能体工具使用循环（agentic tool-use loop）**：LLM 生成响应、可选择发起工具调用、接收结果、继续生成的循环。循环反复进行，直到模型产出不含工具调用的最终文本响应。
- **`tool_use` / `tool_result` 块**：Anthropic API 中，模型想调用工具时会发出一个 `tool_use` 内容块；你的代码执行工具并返回一个 `tool_result` 块。这些块就在普通的消息列表里。
- **嵌入（embedding）**：一串数字（一个向量），用来捕捉一段文本的含义。含义相近的文本对应的向量方向也相近。我们用嵌入来找到与搜索查询匹配的记忆。
- **余弦相似度（cosine similarity）**：衡量两个向量相似程度的方法，计算的是它们夹角的余弦值。1.0 表示方向完全相同，0.0 表示毫无关联。
- **语义检索（semantic search）**：按含义而非精确关键词查找存储内容。我们先对查询做嵌入，再找出余弦相似度最高的已存记忆。
- **智能体自主记忆（agent-driven memory）**：LLM 根据对话上下文自行决定何时存储、检索、修改或丢弃信息，不需要外部启发式规则在每一轮强制写入。


## 架构

<p align="center">
  <img src="../../images/diagrams/23_memory_with_tools.svg" alt="工具化记忆架构图" width="720"/>
</p>

下图展示了完整流程：

1. 用户发送消息。系统提示词里包含全部四个记忆工具的定义。
2. LLM 生成响应。如果它认为记忆相关，就发出一个 `tool_use` 块（例如 `search_memory({"query": "dietary restrictions"})`）。
3. 你的应用对记忆库执行这次工具调用，并返回一个 `tool_result` 块。
4. LLM 看到结果。它可以再次调用工具（链式调用），也可以给出最终的文本响应。
5. 在响应之后，LLM 还可以调用 `save_memory` 存入新事实。这一切都发生在同一轮里。


## 环境准备

安装依赖并配置 API 访问。你需要一个 `LLM_API_KEY` 环境变量（调用 Anthropic API 的密钥）。记忆库中的余弦相似度计算会用到 `numpy`。


In [ ]:
%pip install -q python-dotenv numpy deepagents langchain-openai


导入 Anthropic SDK 和标准库工具。API 密钥从 `.env` 文件加载。


In [ ]:
import os
import json
import uuid
from datetime import datetime, timezone
import numpy as np

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 实现

我们分四步构建记忆即工具系统：

1. 定义四个记忆工具的 schema。
2. 搭建一个支持基于嵌入检索的 `MemoryStore` 后端。
3. 编写一个工具分发器，把工具调用路由到对应的记忆操作。
4. 接好驱动整个系统的智能体循环。


### 第 1 步：定义记忆工具 schema

每个工具 schema 告诉 LLM 这个工具做什么、期望什么输入。Anthropic 的格式使用标准 JSON Schema 的 `input_schema` 字段。清晰的描述能引导模型在合适的时机调用每个工具。


In [ ]:
memory_tools = [
    # OpenAI 工具调用格式：Anthropic 的 input_schema 对应 parameters

    {
        "name": "save_memory",
        "description": (
            "Save an important fact, preference, or piece of information to "
            "long-term memory. Use this when the user shares something worth "
            "remembering for future conversations: personal details, preferences, "
            "goals, or corrections to previous facts."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "content": {
                    "type": "string",
                    "description": "The fact or information to remember.",
                },
                "tags": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Optional category tags (e.g. 'preference', 'personal', 'work').",
                },
            },
            "required": ["content"],
        },
    },
    {
        "name": "search_memory",
        "description": (
            "Search long-term memory for facts relevant to a query. Use this "
            "before answering questions that might benefit from previously stored "
            "context, such as user preferences or past conversations."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query describing what to look for.",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Maximum number of results to return. Defaults to 5.",
                    "default": 5,
                },
            },
            "required": ["query"],
        },
    }
]

现在定义 `update_memory` 和 `delete_memory` 两个工具。
有了它们，智能体就能更正过时的事实，删除用户希望遗忘的信息。
它们与 `save_memory`、`search_memory` 一起，构成一套完整的 CRUD 接口。


In [ ]:
memory_tools += [
    {
        "name": "update_memory",
        "description": (
            "Update an existing memory by its ID. Use this when the user corrects "
            "a previously stored fact or when information changes. Search for the "
            "memory first to get its ID."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "memory_id": {
                    "type": "string",
                    "description": "The ID of the memory to update.",
                },
                "new_content": {
                    "type": "string",
                    "description": "The updated fact or information.",
                },
            },
            "required": ["memory_id", "new_content"],
        },
    },
    {
        "name": "delete_memory",
        "description": (
            "Delete a memory by its ID. Use this when stored information is "
            "no longer true, relevant, or when the user asks you to forget something."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "memory_id": {
                    "type": "string",
                    "description": "The ID of the memory to delete.",
                },
            },
            "required": ["memory_id"],
        },
    },
]

print(f"Defined {len(memory_tools)} memory tools:")
for tool in memory_tools:
    print(f"  - {tool['name']}: {tool['description'][:60]}...")


### 第 2 步：搭建记忆库

想象一个文件柜，每个文件夹的标签都是用隐形墨水写的。要找到对的文件夹，你描述自己要找什么，助手拿你的描述去比对每个标签。这就是语义检索。

我们的 `MemoryStore` 把文本转换成向量（捕捉含义的数字列表）。搜索时，我们对查询做嵌入，找出余弦相似度最高的已存记忆。

这个原型用的是一个轻量的基于哈希的嵌入。在生产环境中，请换成真正的嵌入模型，比如 `voyage-3`、`text-embedding-3-small` 或 sentence-transformer。每条记忆都有一个唯一 ID、一个时间戳、可选的标签，以及它的嵌入向量。


In [ ]:
class MemoryStore:
    """带 CRUD 操作的向量记忆库。"""

    def __init__(self):
        self.memories: dict[str, dict] = {}  # memory_id -> 记录

    def _simple_embed(self, text: str) -> list[float]:
        """用字符级哈希实现的轻量嵌入。

        这是一个原型实现，对短文本能较好地捕捉词语重叠。
        生产环境请换成真正的嵌入模型，例如 voyage-3、
        text-embedding-3-small 或 sentence-transformers。
        """
        dims = 128
        vec = np.zeros(dims)
        words = text.lower().split()
        for word in words:
            # 把每个词哈希到一组维度并累加
            h = hash(word)
            for i in range(3):  # 每个词激活 3 个维度
                idx = abs(h + i * 7919) % dims
                vec[idx] += 1.0
        # 归一化到单位长度
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        return vec.tolist()

    def save(self, content: str, tags: list[str] | None = None) -> str:
        """存入一条新记忆。返回记忆 ID。"""
        memory_id = str(uuid.uuid4())[:8]
        self.memories[memory_id] = {
            "id": memory_id,
            "content": content,
            "tags": tags or [],
            "created_at": datetime.now(timezone.utc).isoformat(),
            "embedding": self._simple_embed(content),
        }
        return memory_id

`search` 方法是语义检索发生的地方。
它先对查询做嵌入，再用余弦相似度（衡量两个向量相似程度的指标）逐一比较所有已存记忆。
结果按相关度排序，最匹配的排在最前。


In [ ]:
def _search(self, query: str, top_k: int = 5) -> list[dict]:
    """找出与查询最相似的记忆。"""
    if not self.memories:
        return []
    query_vec = np.array(self._simple_embed(query))
    scored = []
    for mem in self.memories.values():
        mem_vec = np.array(mem["embedding"])
        # 余弦相似度（向量已归一化）
        similarity = float(np.dot(query_vec, mem_vec))
        scored.append((similarity, mem))
    scored.sort(key=lambda x: x[0], reverse=True)
    results = []
    for score, mem in scored[:top_k]:
        results.append({
            "id": mem["id"],
            "content": mem["content"],
            "tags": mem["tags"],
            "similarity": round(score, 3),
            "created_at": mem["created_at"],
        })
    return results

MemoryStore.search = _search


剩下的 CRUD 方法负责更新、删除和列表。
更新一条记忆时，嵌入会重新计算，这样之后的检索就能反映新内容。
`list_all` 方法会去掉嵌入字段，方便展示。


In [ ]:
def _update(self, memory_id: str, new_content: str) -> bool:
    """更新已有记忆的内容。找到时返回 True。"""
    if memory_id not in self.memories:
        return False
    self.memories[memory_id]["content"] = new_content
    self.memories[memory_id]["embedding"] = self._simple_embed(new_content)
    return True

def _delete(self, memory_id: str) -> bool:
    """按 ID 删除一条记忆。找到时返回 True。"""
    if memory_id not in self.memories:
        return False
    del self.memories[memory_id]
    return True

def _list_all(self) -> list[dict]:
    """返回全部记忆（不含嵌入，用于展示）。"""
    return [
        {"id": m["id"], "content": m["content"], "tags": m["tags"]}
        for m in self.memories.values()
    ]


# 快速测试
store = MemoryStore()
test_id = store.save("Alice is a machine-learning engineer.", tags=["personal"])
print(f"Saved memory with ID: {test_id}")
results = store.search("What does Alice do for work?")
print(f"Search results: {json.dumps(results, indent=2)}")

MemoryStore.update = _update

MemoryStore.delete = _delete

MemoryStore.list_all = _list_all


### 第 3 步：工具分发器

分发器是 LLM 工具调用与记忆库之间的桥梁。当 LLM 发出 `tool_use` 块时，我们读取工具名和输入参数，调用 `MemoryStore` 上对应的方法，并把结果格式化成字符串。


In [ ]:
def dispatch_tool(tool_name: str, tool_input: dict, store: MemoryStore) -> str:
    """执行一次记忆工具调用，返回字符串结果。"""
    if tool_name == "save_memory":
        memory_id = store.save(
            content=tool_input["content"],
            tags=tool_input.get("tags"),
        )
        return json.dumps({"status": "saved", "memory_id": memory_id})

    elif tool_name == "search_memory":
        results = store.search(
            query=tool_input["query"],
            top_k=tool_input.get("top_k", 5),
        )
        return json.dumps({"results": results, "count": len(results)})

    elif tool_name == "update_memory":
        success = store.update(
            memory_id=tool_input["memory_id"],
            new_content=tool_input["new_content"],
        )
        status = "updated" if success else "not_found"
        return json.dumps({"status": status})

    elif tool_name == "delete_memory":
        success = store.delete(memory_id=tool_input["memory_id"])
        status = "deleted" if success else "not_found"
        return json.dumps({"status": status})

    else:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})


# 快速测试：分发一次保存调用
result = dispatch_tool("save_memory", {"content": "Bob likes Thai food", "tags": ["preference"]}, store)
print(f"Dispatch result: {result}")


### 第 4 步：系统提示词

系统提示词引导模型的记忆行为，告诉它何时保存、检索、更新和删除。没有清晰的指令，模型可能无视记忆工具，或者在不合适的时机使用它们。


In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant with long-term memory.

You have four memory tools available:

WHEN TO SAVE:
- Save when the user shares personal facts, preferences, or important details.
- Save corrections to previously known information.
- Do NOT save greetings, small talk, or temporary context.

WHEN TO SEARCH:
- Search before answering questions that might benefit from prior context.
- Search when the user asks "do you remember" or references past conversations.

WHEN TO UPDATE:
- Update when the user corrects a previously stored fact.
- Search first to find the memory ID, then update it.

WHEN TO DELETE:
- Delete when stored information is explicitly wrong or the user asks you to forget.

Keep responses concise. When you use memory tools, briefly mention what you did
(e.g. "I've saved that to memory" or "Let me check my memory")."""

print("System prompt configured.")


### 第 5 步：智能体工具使用循环

想象一通电话，你可以让对方稍等，自己去查点资料。来电者（用户）提出问题，你（智能体）暂停对话，翻查资料，然后回来作答。如果还需要更多信息，可以再次暂停。你给出最终答案时，通话才结束。

智能体循环的运作方式一样。LLM 生成一个响应；如果里面包含 `tool_use` 块，我们就执行工具并把结果回传。循环持续进行，直到模型给出一个以 `end_turn` 结束的最终响应。


In [ ]:
from langchain_core.messages import ToolMessage


def chat_with_memory(
    user_message: str,
    messages: list,
    store: MemoryStore,
) -> tuple[str, list]:
    """把消息送入带记忆工具的智能体循环。

    返回助手的最终文本和更新后的消息列表。
    """
    messages.append({"role": "user", "content": user_message})
    model_with_tools = llm.bind_tools(memory_tools)

    while True:
        response = model_with_tools.invoke(messages)
        # 完整的助手消息（含 tool_calls）进入历史
        messages.append(response)

        if not response.tool_calls:
            return to_text(response.content), messages

        for call in response.tool_calls:
            print(f"  [tool] {call['name']}({json.dumps(call['args'])[:80]})")
            result = dispatch_tool(call["name"], call["args"], store)
            print(f"  [result] {result[:100]}")
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))


print("Agentic loop ready.")


## 运行示例

我们来跑一段多轮对话，把四个记忆操作全部用到。我们要：
1. 分享一些个人信息（智能体应该保存它们）。
2. 问一个需要回忆记忆的问题（智能体应该检索）。
3. 更正一条事实（智能体应该更新）。
4. 让智能体忘掉某件事（智能体应该删除）。


In [ ]:
# 全新的记忆库和对话
store = MemoryStore()
messages = []

exchanges = [
    "Hi! My name is Alice and I'm a machine-learning engineer at Anthropic.",
    "I'm allergic to peanuts and I prefer vegetarian food.",
    "What do you remember about me?",
    "Actually, I moved to a new company. I now work at DeepMind.",
    "Please forget about my food allergy. I'd rather not have that stored.",
    "What do you know about me now?",
]

for user_msg in exchanges:
    print(f"\n{'='*60}")
    print(f"User: {user_msg}")
    print("-" * 60)
    reply, messages = chat_with_memory(user_msg, messages, store)
    print(f"\nAssistant: {reply}")


我们来看看这段对话之后记忆库里有什么。


In [ ]:
print("Current memory store contents:")
print("-" * 40)
for mem in store.list_all():
    print(f"  [{mem['id']}] {mem['content']}")
    if mem['tags']:
        print(f"           tags: {mem['tags']}")
print(f"\nTotal memories: {len(store.memories)}")


### 工具链式调用

这个模式的一个优点是智能体可以在一轮里链式调用多个工具。比如，它先检索一条旧事实，发现已经过时，然后更新它。这一切都发生在给出最终响应之前。我们来看看实际效果。


In [ ]:
# 继续同一段对话
user_msg = "Actually, my name changed too. I go by Ali now, not Alice."
print(f"User: {user_msg}")
print("-" * 60)
reply, messages = chat_with_memory(user_msg, messages, store)
print(f"\nAssistant: {reply}")

print("\nMemory store after update:")
for mem in store.list_all():
    print(f"  [{mem['id']}] {mem['content']}")


## 权衡

### 记忆即工具的优势场景

- **有选择地记忆**：智能体只存重要的东西。食物偏好会被保存，「今天天气怎么样」不会。这让记忆库保持干净、相关。
- **自我更正**：智能体可以在一轮里完成检索、发现过时事实并更新。基于流程的记忆系统常常存入重复或矛盾的内容。
- **透明**：每个记忆操作都体现为一次工具调用。你可以记录、审计、调试智能体到底记了什么、什么时候记的。
- **可组合**：记忆工具与其他工具并列。智能体可以在一轮里检索记忆、调用天气 API 并保存结果。

### 何时会失效

- **LLM 可靠性**：智能体可能忘记保存重要内容，也可能保存过多。记忆管理的质量取决于模型的判断。
- **延迟**：每次工具调用都增加一次往返。如果智能体在回答前链式执行三个记忆操作，就是三次额外的 API 调用。这会增加延迟（从用户提问到最终答案之间的等待时间）。
- **token 成本**：工具定义在每次请求中都要消耗 token。四个工具 schema 大约给每次调用增加 500-800 个 token。对短对话来说，这些开销可能不值得。
- **没有后台处理**：智能体只在活跃轮次里管理记忆。除非你另外构建，否则它不会在对话之间巩固或清理记忆。


## 延伸阅读

- [Anthropic 工具使用文档](https://docs.anthropic.com/en/docs/build-with-claude/tool-use?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：定义工具、处理 `tool_use`/`tool_result` 块以及用 Claude 构建智能体循环的官方指南。
- [OpenAI Function Calling 指南](https://platform.openai.com/docs/guides/function-calling?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：OpenAI 的对应模式，用于在生成过程中向 GPT 模型暴露可调用函数。
- [Packer 等人，"MemGPT: Towards LLMs as Operating Systems," 2023（arXiv:2310.08560）](https://arxiv.org/abs/2310.08560)：把记忆管理当作智能体动作的开创性论文，描述了虚拟记忆层级的显式读写工具。
- [Anthropic：Building Effective Agents（2025）](https://www.anthropic.com/engineering/building-effective-agents?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：生产级智能体的架构模式，包括工具设计原则。
- [Zep 记忆工具集成](https://docs.getzep.com?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：一个生产级记忆服务，把记忆暴露为可通过工具调用的 API，并自动抽取事实。


*← 上一篇：[22 - 多智能体共享记忆](../22_multi_agent_shared_memory/) · 下一篇：[24 - 用 Graphiti 构建图记忆](../24_graph_memory_graphiti/) →*


## 🧪 动手练习

三个小挑战，帮助你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：记忆打标签工具
给 `memory_tools` 增加第五个工具 `tag_memory`，让智能体可以为已存记忆指定主题标签。更新 `dispatch_tool()` 和 `MemoryStore` 以支持标签。跑一段对话，验证智能体确实用标签给记忆做了分类。

### 挑战 2：工具使用频率分析
用 `chat_with_memory()` 跑一段 20 轮的对话。统计智能体调用每个工具（保存、检索、更新、删除）的次数，打印一张汇总表。找出哪些轮触发了记忆保存、哪些触发了检索，以及智能体是否在无人提示的情况下主动更新或删除过记忆。

### 挑战 3：智能体控制 vs. 自动记忆
把同一段 15 轮的对话跑两遍：一遍用本 notebook 的工具化方法，一遍用 07 实体记忆的自动实体抽取。每遍跑完后问 10 个回忆类问题。比较哪种方法存下更多相关事实、哪种方法答对更多问题。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--23-memory-with-tools--memory-with-tools)
